<a href="https://colab.research.google.com/github/walterfegan-byte/Proyectos-redes/blob/main/Copia_de_Copia_de_Pregunta3_yolo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="NQZQqmVvwTcZmqsHcwbw")
project = rf.workspace("helmet-tf").project("safety-helmet-4mhdt")
version = project.version(1)
dataset = version.download("yolov8")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 133.4 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.15
    Uninstalling idna-3.15:
      Successfully uninstalled idna-3.15
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Safety-Helmet-1 in yolov8:: 100%|██████████| 10012/10012 [00:00<00:00, 27487.81it/s]


In [2]:
project = rf.workspace("testcasque").project("ppe-detection-qlq3d")
version = project.version(1)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to PPE-detection-1 in yolov8:: 100%|██████████| 10292/10292 [00:00<00:00, 21417.45it/s]


In [3]:
project = rf.workspace("zayed-uddin-chowdhury-ghymx").project("safety-helmet-wearing-dataset")
version = project.version(3)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Safety-Helmet-Wearing-Dataset-3 in yolov8:: 100%|██████████| 15096/15096 [00:00<00:00, 21701.90it/s]


In [4]:
!pip install ultralytics roboflow -q
import ultralytics
ultralytics.checks()

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
Setup complete ✅ (48 CPUs, 176.9 GB RAM, 48.3/235.7 GB disk)


In [5]:
import os

# Roboflow descargó el dataset en esta ruta
DATASET_PATH = "/content/Safety-Helmet-Wearing-Dataset-3"

# Verificar estructura
for split in ["train", "valid", "test"]:
    img_dir = os.path.join(DATASET_PATH, split, "images")
    lbl_dir = os.path.join(DATASET_PATH, split, "labels")
    n_imgs = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    n_lbls = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
    print(f"{split:6s} → {n_imgs} imágenes | {n_lbls} etiquetas")

# Ver clases del dataset
import yaml
with open(os.path.join(DATASET_PATH, "data.yaml")) as f:
    data_cfg = yaml.safe_load(f)
print("\nClases:", data_cfg["names"])
print("Num clases:", data_cfg["nc"])

train  → 5280 imágenes | 5280 etiquetas
valid  → 1505 imágenes | 1505 etiquetas
test   → 757 imágenes | 757 etiquetas

Clases: ['hat', 'person']
Num clases: 2


Revision de clases de cada uno de los datasets disponibles.

In [6]:
import yaml, os

datasets = {
    "PPE-detection-1":            "/content/PPE-detection-1",
    "Safety-Helmet-1":            "/content/Safety-Helmet-1",
    "Safety-Helmet-Wearing-Dataset-3": "/content/Safety-Helmet-Wearing-Dataset-3",
}

for nombre, ruta in datasets.items():
    yaml_path = os.path.join(ruta, "data.yaml")
    if os.path.exists(yaml_path):
        with open(yaml_path) as f:
            cfg = yaml.safe_load(f)
        print(f"\n📁 {nombre}")
        print(f"   Clases ({cfg['nc']}): {cfg['names']}")
    else:
        print(f"\n❌ {nombre} → no se encontró data.yaml en {ruta}")


📁 PPE-detection-1
   Clases (10): ['boots', 'gloves', 'goggles', 'helmet', 'no-boots', 'no-gloves', 'no-goggles', 'no-helmet', 'no-vest', 'vest']

📁 Safety-Helmet-1
   Clases (3): ['head', 'helmet', 'person']

📁 Safety-Helmet-Wearing-Dataset-3
   Clases (2): ['hat', 'person']


Unificacion de clases

In [7]:
import shutil
from pathlib import Path

# ============================================================
# PASO 1: Define el mapa de clases UNIFICADO
# Edita esto según lo que viste en Celda 1
# ============================================================

# Clases finales del modelo combinado
CLASES_UNIFICADAS = ["helmet", "no_helmet"]  # ajusta según tus datasets

# Mapa: (dataset, clase_original) → clase_unificada
# Si una clase no está aquí, se DESCARTA
MAPA_CLASES = {
    # PPE-detection-1  (ajusta los nombres según lo que veas)
    ("PPE-detection-1", "helmet"):          "helmet",
    ("PPE-detection-1", "hard_hat"):        "helmet",
    ("PPE-detection-1", "no_helmet"):       "no_helmet",
    ("PPE-detection-1", "head"):            "no_helmet",

    # Safety-Helmet-1
    ("Safety-Helmet-1", "helmet"):          "helmet",
    ("Safety-Helmet-1", "Helmet"):          "helmet",
    ("Safety-Helmet-1", "no_helmet"):       "no_helmet",
    ("Safety-Helmet-1", "head"):            "no_helmet",

    # Safety-Helmet-Wearing-Dataset-3
    ("Safety-Helmet-Wearing-Dataset-3", "helmet"):     "helmet",
    ("Safety-Helmet-Wearing-Dataset-3", "no_helmet"):  "no_helmet",
}

# IDs finales
ID_CLASE = {nombre: i for i, nombre in enumerate(CLASES_UNIFICADAS)}
print("Clases unificadas:", ID_CLASE)

# ============================================================
# PASO 2: Crear estructura del dataset combinado
# ============================================================

MERGED_PATH = Path("/content/merged_helmet_dataset")
for split in ["train", "valid", "test"]:
    (MERGED_PATH / split / "images").mkdir(parents=True, exist_ok=True)
    (MERGED_PATH / split / "labels").mkdir(parents=True, exist_ok=True)

print("✅ Directorios creados en", MERGED_PATH)

Clases unificadas: {'helmet': 0, 'no_helmet': 1}
✅ Directorios creados en /content/merged_helmet_dataset


In [8]:
def procesar_dataset(ds_nombre, ds_ruta, mapa, id_clase, merged_path):
    """Copia imágenes y remapea etiquetas al esquema unificado."""

    # Leer clases originales del dataset
    with open(os.path.join(ds_ruta, "data.yaml")) as f:
        cfg = yaml.safe_load(f)
    clases_orig = cfg["names"]  # lista: índice → nombre

    stats = {"copiadas": 0, "etiquetas_ok": 0, "etiquetas_descartadas": 0}

    for split in ["train", "valid", "test"]:
        img_dir = Path(ds_ruta) / split / "images"
        lbl_dir = Path(ds_ruta) / split / "labels"

        if not img_dir.exists():
            continue

        for img_path in img_dir.glob("*.[jp][pn]g"):
            # Nombre único para evitar colisiones entre datasets
            nuevo_nombre = f"{ds_nombre}_{img_path.name}"

            # Copiar imagen
            dest_img = merged_path / split / "images" / nuevo_nombre
            shutil.copy2(img_path, dest_img)
            stats["copiadas"] += 1

            # Procesar etiqueta correspondiente
            lbl_path = lbl_dir / (img_path.stem + ".txt")
            dest_lbl = merged_path / split / "labels" / (nuevo_nombre.rsplit(".", 1)[0] + ".txt")

            if lbl_path.exists():
                nuevas_lineas = []
                for linea in lbl_path.read_text().strip().split("\n"):
                    if not linea.strip():
                        continue
                    partes = linea.split()
                    id_orig = int(partes[0])
                    nombre_orig = clases_orig[id_orig]

                    # Remapear
                    clave = (ds_nombre, nombre_orig)
                    if clave in mapa:
                        nuevo_nombre_clase = mapa[clave]
                        nuevo_id = id_clase[nuevo_nombre_clase]
                        nuevas_lineas.append(f"{nuevo_id} {' '.join(partes[1:])}")
                        stats["etiquetas_ok"] += 1
                    else:
                        stats["etiquetas_descartadas"] += 1  # clase no incluida

                if nuevas_lineas:
                    dest_lbl.write_text("\n".join(nuevas_lineas))
                else:
                    dest_lbl.write_text("")  # imagen sin objetos válidos

    return stats

# Procesar los 3 datasets
for nombre, ruta in datasets.items():
    if os.path.exists(ruta):
        s = procesar_dataset(nombre, ruta, MAPA_CLASES, ID_CLASE, MERGED_PATH)
        print(f"\n✅ {nombre}")
        print(f"   Imágenes copiadas:        {s['copiadas']}")
        print(f"   Anotaciones remapeadas:   {s['etiquetas_ok']}")
        print(f"   Anotaciones descartadas:  {s['etiquetas_descartadas']}")


✅ PPE-detection-1
   Imágenes copiadas:        5140
   Anotaciones remapeadas:   5893
   Anotaciones descartadas:  15692

✅ Safety-Helmet-1
   Imágenes copiadas:        5000
   Anotaciones remapeadas:   24475
   Anotaciones descartadas:  738

✅ Safety-Helmet-Wearing-Dataset-3
   Imágenes copiadas:        7542
   Anotaciones remapeadas:   0
   Anotaciones descartadas:  120279


Integracion de dataset

In [9]:
# Crear data.yaml del dataset combinado
data_yaml = {
    "path":  str(MERGED_PATH),
    "train": "train/images",
    "val":   "valid/images",
    "test":  "test/images",
    "nc":    len(CLASES_UNIFICADAS),
    "names": CLASES_UNIFICADAS,
}

yaml_path = MERGED_PATH / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print("✅ data.yaml creado:", yaml_path)

# Resumen final
print("\n=== RESUMEN DEL DATASET COMBINADO ===")
for split in ["train", "valid", "test"]:
    n = len(list((MERGED_PATH / split / "images").glob("*.[jp][pn]g")))
    print(f"  {split:6s}: {n} imágenes")

✅ data.yaml creado: /content/merged_helmet_dataset/data.yaml

=== RESUMEN DEL DATASET COMBINADO ===
  train : 12377 imágenes
  valid : 3531 imágenes
  test  : 1774 imágenes


In [10]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
    data=str(MERGED_PATH / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    mosaic=1.0,
    flipud=0.0,
    fliplr=0.5,
    hsv_v=0.4,
    degrees=10.0,
    patience=15,
    project="helmet_merged",
    name="yolov8s_3datasets",
    seed=42,
)

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/merged_helmet_dataset/data.yaml, degrees=10.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s_3datasets, nbs=64, nms=False, opset=None, opti

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a8ef1fd5940>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804